# 03. Behavioral Modeling: Purchase Propensity Engine

**Context**: Identifying high-intent users is the cornerstone of e-commerce optimization. In this phase, we analyze user session data to predict the probability of a conversion (purchase).

**The Challenge:** As identified in our EDA, the dataset suffers from a 99:1 class imbalance. Standard models will likely over-fit to the majority "non-purchase" class. To mitigate this, we employ SMOTE (Synthetic Minority Over-sampling Technique) to balance the training set and XGBoost to capture non-linear behavioral signatures.

## 3.1 Pipeline Initialization

We begin by re-loading the raw behavioral stream and applying our modular cleaning and feature engineering logic

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_preprocessing import DataLoader, DataCleaner, FeatureEngineer
from src.behavioral_modeling import BehavioralModel

# Initialize Ingestion & Engineering
loader = DataLoader(data_dir='../data/raw/')
cleaner = DataCleaner()
feat_eng = FeatureEngineer()

# Process Stream
beh_raw, _, _ = loader.load_all()
beh_clean = cleaner.clean_behavioral(beh_raw)
X, y = feat_eng.engineer(beh_clean)

print(f"Features Prepared: {X.shape[1]}")
print(f"Conversion Rate: {y.mean()*100:.2f}%")

Features Prepared: 11
Conversion Rate: 97.88%


## 3.2 Training with Synthetic Balancing

We utilize the BehavioralModel class to perform a Stratified Train-Test Split. This ensures the test set remains a "real-world" representation (imbalanced), while the training set is synthetically balanced via SMOTE to force the model to learn "Buyer" characteristics.

In [2]:
# Initialize Propensity Engine
bm = BehavioralModel()

# Execute Training Loop
# This will print the Resampling stats and the Classification Report
bm.train(X, y)

Original Training Distribution: [ 133 6139]
Training with Balanced Forest (Internal Re-sampling)...

--- Behavioral Propensity Report (Threshold Adjusted) ---
              precision    recall  f1-score   support

           0       0.12      0.06      0.08        33
           1       0.98      0.99      0.99      1535

    accuracy                           0.97      1568
   macro avg       0.55      0.53      0.53      1568
weighted avg       0.96      0.97      0.97      1568

ROC-AUC Score: 0.5745

--- Confusion Matrix (Actual vs Predicted) ---
[[   2   31]
 [  14 1521]]

--- Top Predictive Signals ---
time_on_site        0.146786
avg_session_time    0.146612
bounce_rate         0.144063
age                 0.125650
cart_depth          0.112847
dtype: float64


BalancedRandomForestClassifier(max_depth=10, n_estimators=500, n_jobs=-1,
                               random_state=42)

## 3.3 Evaluation Metrics: Focus on Recall

Because of the imbalance, Accuracy is a deceptive metric. We focus on:

   - Precision/Recall for Class 1: How many buyers did we catch, and how many "window shoppers" did we mislabel?

   -  ROC-AUC: The model's ability to distinguish between classes regardless of the threshold.

## 3.4 Model Persistence

Once satisfied with the Recall scores, we persist the XGBoost model for the final interpretability phase.

In [3]:
# Saves to ../models/propensity_xgb.joblib
bm.save()

☑ Success: High-Sensitivity model saved to ../models/propensity_brf.joblib


In [4]:
X.std()

age                   13.100276
time_on_site           8.280470
pages_viewed           5.373155
previous_purchases     4.239164
cart_items             2.839419
avg_session_time       9.772453
bounce_rate           28.841316
cart_depth             1.285615
engaged                0.498289
discount_seen          0.499948
ad_clicked             0.499971
dtype: float64